# nb02 Clean: tidy the DMHC Financial Summary files for Tableau

**Purpose**
- Clean each of the 8 raw CSVs from nb01 into a Tableau ready CSV in `data/clean/`, one clean file per raw file. No unions and no joins in Python; Tableau wildcard unions the clean files.
- Transformations per file:
  - Parse the 15 measure columns to numbers (strip commas, parenthesized negatives, blanks to nulls).
  - Add a short `Plan` display name next to the legal name.
  - Add period fields: `Period Type` (Annual or Quarterly), `Period End Date` (a real date), `Year`, `Quarter`, and `Quarter Label` like `2025 Q4`.
- Derived measures (per member per month, margin, Medical Loss Ratio comparisons) are intentionally NOT computed here; they will be Tableau calculated fields.

**Prerequisites**
- Python 3 with `pandas`. Run from the `tableau_plan_financials/notebooks/` folder, after nb01.

**Outputs**
- `data/clean/financial_summary_<plan>_<window>.csv`, 8 files.
- Diagnostics in `nb02_clean_cell_output.txt`, including a live checked control total.

In [1]:
# Step 0: mirror all printed output to a text file for easy sharing
import sys
from pathlib import Path

SINK_PATH = Path.cwd() / "nb02_clean_cell_output.txt"

_orig_out = getattr(sys, "_nb_orig_stdout", sys.stdout)
_orig_err = getattr(sys, "_nb_orig_stderr", sys.stderr)
sys._nb_orig_stdout, sys._nb_orig_stderr = _orig_out, _orig_err

class _Tee:
    def __init__(self, stream, fh):
        self.stream, self.fh = stream, fh
    def write(self, data):
        self.stream.write(data)
        self.fh.write(data)
        self.fh.flush()
    def flush(self):
        self.stream.flush()
        self.fh.flush()

_sink = open(SINK_PATH, "w")
sys.stdout = _Tee(_orig_out, _sink)
sys.stderr = _Tee(_orig_err, _sink)
print(f"Mirroring cell output to {SINK_PATH.name} (attach this file in the chat)")

Mirroring cell output to nb02_clean_cell_output.txt (attach this file in the chat)
8 raw files expected; missing: none
Helpers ready.
[financial_summary_lacare_2010_2014.csv] 25 rows in -> 25 rows out, 27 columns, parse failures: 0 -> financial_summary_lacare_2010_2014.csv
[financial_summary_lacare_2015_2019.csv] 25 rows in -> 25 rows out, 27 columns, parse failures: 0 -> financial_summary_lacare_2015_2019.csv
[financial_summary_lacare_2020_2023.csv] 20 rows in -> 20 rows out, 27 columns, parse failures: 0 -> financial_summary_lacare_2020_2023.csv
[financial_summary_lacare_2024_2026.csv] 11 rows in -> 11 rows out, 27 columns, parse failures: 0 -> financial_summary_lacare_2024_2026.csv
[financial_summary_hncs_2010_2014.csv] 25 rows in -> 25 rows out, 27 columns, parse failures: 0 -> financial_summary_hncs_2010_2014.csv
[financial_summary_hncs_2015_2019.csv] 25 rows in -> 25 rows out, 27 columns, parse failures: 0 -> financial_summary_hncs_2015_2019.csv
[financial_summary_hncs_2020_2023.

In [2]:
# Step 1: setup and the file plan
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

PLANS = {
    "lacare": "L.A. Care",                       # license 933 0355, Local Initiative Health Authority for Los Angeles County
    "hncs": "Health Net Community Solutions",    # license 933 0426
}
WINDOWS = ["2010_2014", "2015_2019", "2020_2023", "2024_2026"]

MEASURE_COLS = ["Total Enrollees", "Medi-Cal Managed Care", "TNE", "Required TNE", "Excess TNE",
                "% TNE to Required", "Total Assets", "Total Current Assets",
                "Total Current Liabilities", "Total Revenue", "Net Income-Loss",
                "Total Administrative Expenses", "Administrative Cost Ratio",
                "Total Medical Expenses", "Health Expense Ratio"]

FILES = [(tag, window, RAW_DIR / f"financial_summary_{tag}_{window}.csv")
         for tag in PLANS for window in WINDOWS]
missing = [f.name for _, _, f in FILES if not f.exists()]
print(f"{len(FILES)} raw files expected; missing: {missing if missing else 'none'}")

In [3]:
# Step 2: cleaning helpers
def parse_number(x):
    """Strip commas and percent signs, treat parentheses as negative, blanks as null."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().replace(",", "").replace("%", "").replace("$", "")
    if s in ("", "-", "N/A", "NA"):
        return pd.NA
    neg = s.startswith("(") and s.endswith(")")
    if neg:
        s = s[1:-1]
    try:
        v = float(s)
    except ValueError:
        return pd.NA
    return -v if neg else v

def clean_file(tag, window, path):
    """Clean one raw CSV; returns the cleaned frame and a parse failure count."""
    df = pd.read_csv(path, dtype=str)
    n_in = len(df)

    # numeric measures
    failures = 0
    for col in MEASURE_COLS:
        raw_nonblank = df[col].notna() & (df[col].astype(str).str.strip() != "")
        df[col] = df[col].map(parse_number)
        failures += int((raw_nonblank & df[col].isna()).sum())

    # plan display name
    df.insert(1, "Plan", PLANS[tag])

    # period fields
    df["Period End Date"] = pd.to_datetime(df["Statement Date"], format="%m/%d/%Y")
    df["Period Type"] = df["Report Type"].str.extract(r"^(Annual|Quarterly)")[0]
    df["Year"] = df["Period End Date"].dt.year
    df["Quarter"] = df["Period End Date"].dt.quarter
    df["Quarter Label"] = df["Year"].astype(str) + " Q" + df["Quarter"].astype(str)
    # an annual row represents the full year, not a quarter
    df.loc[df["Period Type"] == "Annual", "Quarter"] = pd.NA
    df["Quarter"] = df["Quarter"].astype("Int64")
    df.loc[df["Period Type"] == "Annual", "Quarter Label"] = df["Year"].astype(str) + " Annual"
    df["Period End Date"] = df["Period End Date"].dt.strftime("%Y-%m-%d")

    out = CLEAN_DIR / path.name
    df.to_csv(out, index=False)
    print(f"[{path.name}] {n_in} rows in -> {len(df)} rows out, {df.shape[1]} columns, "
          f"parse failures: {failures} -> {out.name}")
    return df, failures

print("Helpers ready.")

In [4]:
# Step 3: clean every file
frames = {}
total_failures = 0
for tag, window, path in FILES:
    if not path.exists():
        print(f"skipping missing {path.name}")
        continue
    frames[(tag, window)], f = clean_file(tag, window, path)
    total_failures += f
print(f"\nCleaned {len(frames)} files; total numeric parse failures: {total_failures}")

In [5]:
# Step 4: verification (checks only; the clean files stay separate for Tableau)
# a temporary concatenation is used here purely to verify, it is not written anywhere
chk = pd.concat(frames.values(), ignore_index=True)
chk["Period End Date"] = pd.to_datetime(chk["Period End Date"])

print("=== coverage ===")
for plan, g in chk.groupby("Plan"):
    q = g[g["Period Type"] == "Quarterly"]
    a = g[g["Period Type"] == "Annual"]
    print(f"[{plan}] {len(g)} rows; quarterly {len(q)} ({q['Period End Date'].min().date()} "
          f"to {q['Period End Date'].max().date()}); annual {len(a)} "
          f"({a['Year'].min()} to {a['Year'].max()})")

print("\n=== control total: L.A. Care 2025 annual Total Revenue ===")
v = chk[(chk["Plan"] == "L.A. Care") & (chk["Period Type"] == "Annual") & (chk["Year"] == 2025)]["Total Revenue"]
got = float(v.iloc[0]) if len(v) else None
expected = 15800808336.0  # verified live against the DMHC report during diagnostics
print(f"expected {expected:,.0f}; got {got:,.0f}; match: {got == expected}" if got is not None
      else "row not found")

print("\n=== loss quarters (Net Income-Loss below zero, quarterly reports) ===")
losses = chk[(chk["Period Type"] == "Quarterly") & (chk["Net Income-Loss"] < 0)]
for plan, g in losses.groupby("Plan"):
    labels = g.sort_values("Period End Date")["Quarter Label"].tolist()
    print(f"[{plan}] {len(g)} loss quarters of 65: {labels}")

print("\n=== ratio sanity (quarterly Health Expense Ratio) ===")
for plan, g in chk[chk["Period Type"] == "Quarterly"].groupby("Plan"):
    r = g["Health Expense Ratio"].dropna()
    print(f"[{plan}] min {r.min():.2f}, median {r.median():.2f}, max {r.max():.2f}, "
          f"missing {g['Health Expense Ratio'].isna().sum()}")

print("\n=== null counts in key measures ===")
key = ["Total Revenue", "Net Income-Loss", "Total Medical Expenses", "Total Administrative Expenses",
       "Medi-Cal Managed Care", "TNE", "Required TNE"]
print(chk[key].isna().sum().to_string())

In [6]:
# Step 5: confirm the output sink
sys.stdout.flush()
print(f"\nAll printed output saved to: {SINK_PATH}")
print(f"File size: {SINK_PATH.stat().st_size:,} bytes")

**Next step**
- Attach `nb02_clean_cell_output.txt` in the chat.
- If the control total matches and the parse failures are zero, the data preparation is done and the Tableau build starts: wildcard union of `data/clean/financial_summary_*.csv`, one step at a time.
- The loss quarter and ratio printouts preview the story before any chart exists.